# 01 – Distill Student 1D-CNN (Kaggle)

Run this notebook on **Kaggle (T4 x2 recommended)** when no student checkpoint exists yet.

**Flow**
```
/kaggle/input/<payload-dataset>/
    payload_256.npy + metadata.csv          ← upload from local step 01
    ↓
build_teacher_targets (SecureBERT fp16)     → teacher_targets.npy
    ↓
train_student_cnn (DDP on 2 GPUs)          → student_cnn_best.pt
    ↓
export_student_embeddings                   → student_embeddings.npy
    ↓
/kaggle/working/student_results.zip        ← download this
```

**After downloading `student_results.zip`**
- Extract `data/processed/student_cnn_best.pt` → `models/student_cnn_best.pt` (local)
- Extract `data/processed/student_embeddings.npy` → `data/processed/` (local)
- Then run local notebooks `02_export_student_embeddings.ipynb` can be skipped;
  go directly to `03_build_three_tier_graph.ipynb`.

In [ ]:
from pathlib import Path
import torch

# ── Repo ───────────────────────────────────────────────────────────────────────
GITHUB_REPO_URL       = "https://github.com/LeThanhPhat-ATTT2023/Do-an-chuyen-nganh_NT114.git"
GITHUB_BRANCH         = ""        # empty = default branch
FORCE_RECLONE         = False
GITHUB_PULL_IF_CLONED = True

WORK_DIR   = Path("/kaggle/working/nt114_student")
RESULT_ZIP = Path("/kaggle/working/student_results.zip")

# ── Input filenames (must match what was uploaded to Kaggle Dataset) ────────────
PAYLOAD_NPY_NAME  = "payload_256.npy"
METADATA_CSV_NAME = "metadata.csv"

# ── Teacher (SecureBERT) ───────────────────────────────────────────────────────
TEACHER_MODEL_NAME = "ehsanaghaei/SecureBERT"
TEACHER_BATCH_SIZE = 256        # per GPU; 256 works on T4 16 GB with fp16
TEACHER_MAX_LENGTH = 512
TEACHER_SAVE_DTYPE = "float16"  # halves disk usage; negligible cosine loss

# ── Student training ───────────────────────────────────────────────────────────
STUDENT_EPOCHS     = 30
STUDENT_BATCH_SIZE = 1024       # per GPU; DDP scales effective batch automatically
STUDENT_LR         = 1e-3
STUDENT_PATIENCE   = 6

# ── Student inference ──────────────────────────────────────────────────────────
STUDENT_INFER_BATCH = 2048

# ── Install / GPU ──────────────────────────────────────────────────────────────
INSTALL_WITH_DEPS   = False
INSTALL_MISSING_DEPS= True

GPU_IDS: list[int] = list(range(torch.cuda.device_count())) or [0]

In [ ]:
import sys, torch

print("Python:", sys.version)
print("Torch:",  torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  VRAM={p.total_memory/1024**3:.1f} GB")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available.\n"
        "Kaggle: Session options → Accelerator → GPU T4 x2"
    )

print(f"\nAvailable GPU_IDS: {GPU_IDS}")

In [ ]:
# ── Repo setup + install ───────────────────────────────────────────────────────
import os, shutil, subprocess, sys
from pathlib import Path


def run_streaming(cmd, *, cwd=None, env=None):
    """Run a subprocess and stream output line-by-line; progress bars update in-place."""
    print("\n$", " ".join(str(x) for x in cmd))
    proc = subprocess.Popen(
        [str(x) for x in cmd], cwd=cwd or WORK_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        encoding='utf-8', errors='replace', bufsize=1,
    )
    _prev_progress = False
    for line in proc.stdout:
        text = line.rstrip('\n\r')
        is_progress = '%|' in text
        if is_progress:
            sys.stdout.write('\r' + text if _prev_progress else text)
        else:
            if _prev_progress:
                sys.stdout.write('\n')
            sys.stdout.write(text + '\n')
        sys.stdout.flush()
        _prev_progress = is_progress
    if _prev_progress:
        sys.stdout.write('\n')
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


# Keep run_cmd as alias for setup/install calls (git, pip, etc.)
run_cmd = run_streaming


def is_repo_root(path):
    return (
        (path / "src" / "graphslm_ids").exists()
        and (path / "pyproject.toml").exists()
    )


def find_repo_in_kaggle_input():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for root in sorted(input_root.glob("*")):
        if is_repo_root(root):
            return root
        for p in root.rglob("pyproject.toml"):
            cand = p.parent
            if is_repo_root(cand):
                return cand
    return None


def prepare_repo():
    if FORCE_RECLONE and WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    if is_repo_root(WORK_DIR):
        if GITHUB_PULL_IF_CLONED and (WORK_DIR / ".git").exists():
            try:
                subprocess.check_call(["git", "pull", "--ff-only"], cwd=WORK_DIR)
            except subprocess.CalledProcessError as e:
                print(f"[warn] git pull failed ({e}), continuing with existing code.")
        return
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    source = find_repo_in_kaggle_input()
    if source:
        print("Copy repo:", source, "->", WORK_DIR)
        shutil.copytree(
            source, WORK_DIR,
            ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", ".pytest_cache")
        )
        return
    cmd = ["git", "clone"]
    if GITHUB_BRANCH.strip():
        cmd += ["--branch", GITHUB_BRANCH]
    cmd += [GITHUB_REPO_URL, str(WORK_DIR)]
    subprocess.check_call(cmd)


def import_ok(name):
    try:
        __import__(name)
        return True
    except Exception:
        return False


def install_repo():
    missing = [
        n for n in ["numpy", "pandas", "yaml", "torch", "tqdm", "transformers"]
        if not import_ok(n)
    ]
    if missing:
        if not INSTALL_MISSING_DEPS:
            raise RuntimeError(f"Missing deps: {missing}")
        run_streaming([sys.executable, "-m", "pip", "install", "-r", "requirements-ml.txt"])
    cmd = [sys.executable, "-m", "pip", "install", "-e", "."]
    if not INSTALL_WITH_DEPS:
        cmd.append("--no-deps")
    run_streaming(cmd)


prepare_repo()
os.chdir(WORK_DIR)
install_repo()
print("CWD:", Path.cwd())

In [ ]:
# ── Locate input files ─────────────────────────────────────────────────────────
from pathlib import Path


def find_input_file(name):
    for p in sorted(Path("/kaggle/input").rglob(name)):
        return p
    raise FileNotFoundError(
        f"{name!r} not found in /kaggle/input.\n"
        "Upload payload_256.npy + metadata.csv as a Kaggle Dataset."
    )


PAYLOAD_NPY  = find_input_file(PAYLOAD_NPY_NAME)
METADATA_CSV = find_input_file(METADATA_CSV_NAME)

PROCESSED_DIR = WORK_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TEACHER_NPY  = PROCESSED_DIR / "teacher_targets.npy"
STUDENT_CKPT = PROCESSED_DIR / "student_cnn_best.pt"
STUDENT_EMB  = PROCESSED_DIR / "student_embeddings.npy"

print("PAYLOAD_NPY :", PAYLOAD_NPY)
print("METADATA_CSV:", METADATA_CSV)

import numpy as np
arr = np.load(PAYLOAD_NPY, mmap_mode="r")
print(f"payload_256.npy: {arr.shape}  dtype={arr.dtype}")

In [ ]:
# ── Step 1: Build teacher targets (SecureBERT) ─────────────────────────────────
import subprocess, sys

if TEACHER_NPY.exists():
    print("Teacher targets already exist:", TEACHER_NPY, "— skipping.")
else:
    cmd = [
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.preprocessing.build_teacher_targets",
        "--payload-npy",   str(PAYLOAD_NPY),
        "--metadata-csv",  str(METADATA_CSV),
        "--output-path",   str(TEACHER_NPY),
        "--model-name",    TEACHER_MODEL_NAME,
        "--batch-size",    str(TEACHER_BATCH_SIZE),
        "--max-length",    str(TEACHER_MAX_LENGTH),
        "--save-dtype",    TEACHER_SAVE_DTYPE,
        "--device",        "auto",
        "--compile",
    ]
    run_streaming(cmd, cwd=WORK_DIR)

import numpy as np
t = np.load(TEACHER_NPY, mmap_mode="r")
print(f"teacher_targets.npy: {t.shape}  dtype={t.dtype}")

In [ ]:
# ── Step 2: Train student 1D-CNN via distillation ──────────────────────────────
import subprocess, sys, shutil
from pathlib import Path

STUDENT_OUTPUT_DIR = WORK_DIR / "outputs" / "student_cnn"

if STUDENT_CKPT.exists():
    print("Student checkpoint already exists:", STUDENT_CKPT, "— skipping training.")
else:
    n_gpus = len(GPU_IDS)
    if n_gpus > 1:
        cmd = [
            "torchrun", "--standalone", f"--nproc_per_node={n_gpus}",
            "-m", "graphslm_ids.offline_path.training.train_student_cnn",
        ]
    else:
        cmd = [sys.executable, "-u", "-m",
               "graphslm_ids.offline_path.training.train_student_cnn"]

    cmd += [
        "--payload-npy",  str(PAYLOAD_NPY),
        "--teacher-npy",  str(TEACHER_NPY),
        "--output-dir",   str(STUDENT_OUTPUT_DIR),
        "--epochs",       str(STUDENT_EPOCHS),
        "--batch-size",   str(STUDENT_BATCH_SIZE),
        "--lr",           str(STUDENT_LR),
        "--patience",     str(STUDENT_PATIENCE),
        "--compile",
    ]
    run_streaming(cmd, cwd=WORK_DIR)

    best_ckpt_in_output = STUDENT_OUTPUT_DIR / "student_cnn_best.pt"
    if best_ckpt_in_output.exists():
        shutil.copy2(best_ckpt_in_output, STUDENT_CKPT)
        print("Copied best checkpoint:", STUDENT_CKPT)
    else:
        raise FileNotFoundError(
            f"Expected checkpoint not found: {best_ckpt_in_output}\n"
            "Check training logs for errors."
        )

print("Student checkpoint:", STUDENT_CKPT, f"({STUDENT_CKPT.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# ── Step 3: Export student embeddings ──────────────────────────────────────────
import subprocess, sys

if STUDENT_EMB.exists():
    print("Student embeddings already exist:", STUDENT_EMB, "— skipping.")
else:
    cmd = [
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.training.export_student_embeddings",
        "--payload-npy",  str(PAYLOAD_NPY),
        "--checkpoint",   str(STUDENT_CKPT),
        "--output-path",  str(STUDENT_EMB),
        "--batch-size",   str(STUDENT_INFER_BATCH),
        "--l2-normalize",
        "--device",       "auto",
    ]
    run_streaming(cmd, cwd=WORK_DIR)

import numpy as np
emb = np.load(STUDENT_EMB, mmap_mode="r")
print(f"student_embeddings.npy: {emb.shape}  dtype={emb.dtype}")

In [ ]:
# ── Step 4: Bundle results for download ────────────────────────────────────────
import json, zipfile
from pathlib import Path

if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()

files_to_bundle = [
    STUDENT_CKPT,
    STUDENT_EMB,
    PROCESSED_DIR / "teacher_targets.meta.json",
]

# Training summary if it exists.
for p in (WORK_DIR / "outputs" / "student_cnn").rglob("training_summary.json"):
    files_to_bundle.append(p)

with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    for p in files_to_bundle:
        p = Path(p)
        if not p.exists():
            print(f"  [skip] {p.name} not found")
            continue
        try:
            arcname = p.relative_to(WORK_DIR)
        except ValueError:
            arcname = Path(p.name)
        zf.write(p, arcname)
        print(f"  + {arcname}  ({p.stat().st_size / 1e6:.1f} MB)")

print(f"\nResult ZIP : {RESULT_ZIP}")
print(f"Total size : {RESULT_ZIP.stat().st_size / 1e6:.1f} MB")
print("\nDownload this file, then extract:")
print("  data/processed/student_cnn_best.pt       → <project>/models/")
print("  data/processed/student_embeddings.npy    → <project>/data/processed/")
print("\nIf student_embeddings.npy is large, you can also upload it directly")
print("to a Kaggle Dataset and skip the local export step.")

## Notes

- **Required Kaggle input**: upload `payload_256.npy` + `metadata.csv` (from local step 01)
  as a Kaggle Dataset before running this notebook.
- **Teacher model**: `ehsanaghaei/SecureBERT` (~440 MB) is downloaded from HuggingFace on
  first run. Subsequent runs skip this step.
- **DDP**: when 2 GPUs are available, student training uses `torchrun --standalone` for DDP,
  doubling effective throughput.
- **`TEACHER_SAVE_DTYPE="float16"`** halves disk usage for teacher targets with negligible
  impact on cosine-similarity distillation loss.
- **Re-running**: all three steps check for existing output files and skip if found.
  Use `FORCE_RECLONE=True` to re-clone the repo on re-runs.
- **Large student embeddings**: `student_embeddings.npy` can be very large (≥10 GB).
  You can upload it directly to a Kaggle Dataset instead of downloading it in the ZIP,
  then feed it straight into the HGT graph-build step on Kaggle if preferred.